
Jeremy Granflaten 1/18/2026

Need to start getting data for the milestone project. The goal is to see if my idea is going to work
for this project.
Started with multiple pulls to get specific financial data from the EDGAR database from the SEC. Once I was connected and receiving information, I noticed there was no particular heading for each section. I ended up pulling in the discourses as a full text to use with the prompt.

Want to create an easy-to-read summary for normal people with no training on these discourses to understand. It should be
able to identify risks, trends, business performance, and any other important information for the company selected for educational
purposes, and specifically not for investment. If any investment recommendations are made, it would open this up to regulatory
or ethical problems.y

In [4]:

import os
import requests
import re
import textstat
import time
import json
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

BASE_MODEL = "gpt-3.5-turbo-0125"

print("Setup complete.")


Setup complete.



Getting the financial disclosures from the EDGAR database. I there were no specific headings for sections that
were consistent, when doing a test for Apple, I decided to get the primary document for the 10-K. This file is a must to be
submitted and should have all the information I am looking for to do the summarization. For this, you would have to get the Cik number for the company you want to look at.

In [7]:

headers = {
    "User-Agent": "FinancialDisclosureResearchProject"
}

cik = "0000320193"

submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"

submissions = requests.get(submissions_url, headers=headers).json()

filings = submissions["filings"]["recent"]

accession = None
primary_doc = None

for form, acc, doc in zip(
    filings["form"],
    filings["accessionNumber"],
    filings["primaryDocument"]
):
    if form == "10-K":
        accession = acc.replace("-", "")
        primary_doc = doc
        break


filing_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{primary_doc}"

print("Filing URL:", filing_url)

response = requests.get(filing_url, headers=headers)

filing_text = response.text

print("Filing length:", len(filing_text))

Filing URL: https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm
Filing length: 1520208



In my first try, I wanted to find the MD&A information and thought it would be specifically disclosed.
I found out that this wasn't the case and couldn't rely on finding this exact match. I kept getting nothing
to come back. I have to get this information without depending on the specific headers or narratives stating
this information.

In [10]:

def extract_narrative_financial_text(text, min_length=4000):

    cleaned = re.sub(r'\s+', ' ', text)

    return cleaned[:min_length]


analysis_text = extract_narrative_financial_text(filing_text)
print("Narrative Length:", len(analysis_text))

Narrative Length: 4000



I started with these five prompts. These prompts look to summarize the information into plain language
output that is readable for anyone. It should be accurate regarding what risks, trends, and any other needed
information were given in the disclosure. They are being told not to put it into a form that offers any investment
advice on the company. These prompts will have to be fine-tuned to get better responses in later iterations of this project.

In [13]:

prompts = [
    "Explain the following financial disclosure in plain language for a non-expert reader.",
    "Summarize the financial performance and risks discussed below without offering advice.",
    "Rewrite this disclosure so a first-time investor could understand it.",
    "Extract the key risks and trends described, using simple and neutral language.",
    "Provide a factual explanation of what this disclosure says about the company’s performance."
]

for i, prompt in enumerate(prompts, 1):
    print(f"\nPROMPT {i}:\n{prompt}\n")


PROMPT 1:
Explain the following financial disclosure in plain language for a non-expert reader.


PROMPT 2:
Summarize the financial performance and risks discussed below without offering advice.


PROMPT 3:
Rewrite this disclosure so a first-time investor could understand it.


PROMPT 4:
Extract the key risks and trends described, using simple and neutral language.


PROMPT 5:
Provide a factual explanation of what this disclosure says about the company’s performance.




Creating a baseline model response. Started with a newer version of GPT, but later in the process, it errored out, so I had to find the version GPT-3.5-turbo-0125 that would work for the rest of the process. These responses will help with fine-tuning later to get better responses. Making sure it knows what kind of data it is looking at with the financial disclosure simplification assistant to help make the output simpler.

In [16]:

baseline_results = {}

for i, prompt in enumerate(prompts, 1):

    full_prompt = f"{prompt}\n\nFinancial Disclosure:\n{analysis_text}"

    response = client.chat.completions.create(

        model=BASE_MODEL,

        messages=[

            {"role": "system", "content": "You are a financial disclosure simplification assistant."},

            {"role": "user", "content": full_prompt}

        ],

        temperature=0.3

    )

    output = response.choices[0].message.content

    baseline_results[f"Prompt_{i}"] = output


    print("\n" + "="*60)
    print(f"BASELINE PROMPT {i}")
    print("="*60)
    print(output)


BASELINE PROMPT 1
This financial disclosure document contains information about a company's financial activities and performance. It includes details such as revenue expectations, debt obligations, asset values, and liabilities. The document is structured using a specific format called XBRL, which helps organize and present financial data for analysis and reporting purposes.

BASELINE PROMPT 2
The financial performance and risks discussed in the document include revenue recognition, long-term debt, operating lease assets, finance lease assets, operating lease liabilities, and risks associated with these financial elements.

BASELINE PROMPT 3
Financial Disclosure:

This document provides information about a company's financial performance for the year 2025. The company's central index key is 0000320193. It includes details about revenue expectations and liabilities related to long-term debt, operating leases, and finance leases.

BASELINE PROMPT 4
Key Risks and Trends:
- Uncertainty in

Since the goal is to make the output easier for a normal person to understand, I wanted to evaluate it. I found a way to use Flesch Reading Ease
and Flesch-Kincaid grade level to see how easy and what grade level the output would be. I wanted to see if fine-tuning would help later.

In [19]:

print("\nBASELINE READABILITY METRICS")

baseline_readability = {}

for key, text in baseline_results.items():
    reading = textstat.flesch_reading_ease(text)
    grade = textstat.flesch_kincaid_grade(text)
    
    baseline_readability[key] = {

        "reading": reading,
        "grade": grade

    }

    print("\n", key)
    print("Reading Ease:", reading)
    print("Grade Level:", grade)


BASELINE READABILITY METRICS

 Prompt_1
Reading Ease: 7.295000000000016
Grade Level: 16.216666666666665

 Prompt_2
Reading Ease: -6.914999999999964
Grade Level: 21.676666666666666

 Prompt_3
Reading Ease: 17.932307692307717
Grade Level: 13.98769230769231

 Prompt_4
Reading Ease: -20.99928571428569
Grade Level: 23.14428571428572

 Prompt_5
Reading Ease: 12.743095238095265
Grade Level: 17.27857142857143


I don't want this information to be used as investment advice. I wanted to see if some words are in the output. The first time I ran this, I got a response with the words 'invest' included in prompts 1 and 3. There is a slight change every time it is run, but this is a good test.

In [22]:

advice_keywords = [
    "buy",
    "sell",
    "invest",
    "should purchase",
    "strong opportunity",
    "recommend"
]

baseline_advice = {}

print("\nBASELINE ADVICE CHECK")

for key, text in baseline_results.items():
    violations = [
        word for word in advice_keywords
        if word in text.lower()
    ]

    baseline_advice[key] = violations


    print("\n", key)
    if violations:
        print("Advice detected:", violations)
    else:
        print("No advice detected")


BASELINE ADVICE CHECK

 Prompt_1
No advice detected

 Prompt_2
No advice detected

 Prompt_3
No advice detected

 Prompt_4
No advice detected

 Prompt_5
No advice detected


Creating the JSON training data for the fine-tuning. Had to include at least 10 examples for it to work. I tried with less, and it errored out later because there wasn't enough for it to fine-tune with. Making sure it is in plain language for everyone to understand, and that there is no investment information or comments.

In [25]:

training_examples = []

sample_pairs = [
    ("Net sales increased 8% year-over-year due to higher iPhone revenue and services growth.",
     "The company earned more money than last year because it sold more iPhones and its services business grew."),
    
    ("Gross margin declined due to increased component costs and foreign exchange impacts.",
     "The company made slightly less profit on each product because parts became more expensive and currency changes increased costs."),
    
    ("Operating expenses increased primarily due to research and development investments.",
     "The company spent more money on research and development to create new products and improve existing ones."),
    
    ("Cash flow from operations improved compared to the prior fiscal year.",
     "The company generated more cash from its main business activities than it did last year."),
    
    ("Revenue growth was partially offset by supply chain constraints.",
     "Although the company made more money, supply chain problems reduced how much it could grow."),
    
    ("The company returned value to shareholders through dividends and share repurchases.",
     "The company gave money back to shareholders by paying dividends and buying back stock."),
    
    ("R&D investments focused on developing new products and services.",
     "The company spent money on research to create new products and improve services."),
    
    ("Supply chain disruptions impacted delivery schedules.",
     "Problems in the supply chain caused some delays in delivering products."),
    
    ("The company maintained strong liquidity throughout the fiscal year.",
     "The company had enough cash and assets to cover its expenses throughout the year."),
    
    ("Operating income declined slightly due to higher costs of goods sold.",
     "The company's profit from operations decreased a little because product costs went up.")
]

for tech, plain in sample_pairs:

    training_examples.append({

        "messages": [

            {"role": "system", "content": "Provide neutral plain language explanation. No investment advice."},

            {"role": "user", "content": tech},

            {"role": "assistant", "content": plain}

        ]

    })


with open("train.jsonl", "w") as f:

    for example in training_examples:

        f.write(json.dumps(example) + "\n")


print("Training file saved.")

Training file saved.


Uploading the JSON to OpenAI to fine-tune.

In [28]:

upload = client.files.create(
file=open("train.jsonl","rb"),
purpose="fine-tune"
)

print(upload.id)

file-PezpELU7q1iYVx6Rex2Q8V


In [30]:

job = client.fine_tuning.jobs.create(
training_file=upload.id,
model=BASE_MODEL
)

job_id = job.id

print(job_id)

ftjob-gsC1fbVW6L6FWYkKQiZIEKGu


polls the fine-tuning job until it succeeds or fails. Making sure to track it to see when or if something goes wrong. Make sure it succeededs before moving on.

In [33]:
while True:

    status = client.fine_tuning.jobs.retrieve(job_id)

    print(datetime.now(), status.status)

    if status.status in ["succeeded","failed"]:

        break

    time.sleep(20)


FINE_TUNED_MODEL = status.fine_tuned_model

print("MODEL:", FINE_TUNED_MODEL)

2026-03-02 19:31:55.240721 running
2026-03-02 19:32:15.483229 running
2026-03-02 19:32:35.738776 running
2026-03-02 19:32:56.520655 running
2026-03-02 19:33:16.789192 running
2026-03-02 19:33:37.077104 running
2026-03-02 19:33:57.355710 running
2026-03-02 19:34:17.616355 running
2026-03-02 19:34:37.868757 running
2026-03-02 19:34:58.517289 running
2026-03-02 19:35:19.780236 running
2026-03-02 19:35:40.027494 running
2026-03-02 19:36:00.301974 running
2026-03-02 19:36:20.521108 running
2026-03-02 19:36:40.734015 running
2026-03-02 19:37:01.055392 running
2026-03-02 19:37:21.344905 running
2026-03-02 19:37:41.621405 running
2026-03-02 19:38:01.869988 running
2026-03-02 19:38:22.141856 running
2026-03-02 19:38:42.444823 running
2026-03-02 19:39:02.709443 running
2026-03-02 19:39:22.980462 running
2026-03-02 19:39:43.204541 running
2026-03-02 19:40:03.459047 running
2026-03-02 19:40:23.727218 running
2026-03-02 19:40:43.982854 running
2026-03-02 19:41:04.263966 running
2026-03-02 19:41:24.

I put this check in just to make sure everything went well and some information on the model to help and feel comfortable.

In [36]:
events = client.fine_tuning.jobs.list_events(job_id)

for event in events.data:
    print(event.message)

The job has successfully completed
Usage policy evaluations completed, model is now enabled for sampling
Moderation checks for snapshot ft:gpt-3.5-turbo-0125:personal-api::DF8yXyYT passed.
Evaluating model against our usage policies
New fine-tuned model created
Checkpoint created at step 90
Checkpoint created at step 80
Step 100/100: training loss=0.00
Step 99/100: training loss=0.00
Step 98/100: training loss=0.00
Step 97/100: training loss=0.00
Step 96/100: training loss=0.00
Step 95/100: training loss=0.00
Step 94/100: training loss=0.00
Step 93/100: training loss=0.00
Step 92/100: training loss=0.00
Step 91/100: training loss=0.00
Step 90/100: training loss=0.00
Step 89/100: training loss=0.00
Step 88/100: training loss=0.00


I wanted to run the same prompts on the fine-tune data to see the outputs and compare them later to make sure the fine-tuning worked. As you can see, the output is shorter than the original one. Leads me to think it is in a more natural language and straightforward.

In [39]:
ft_results = {}

for i, prompt in enumerate(prompts,1):

    full_prompt = f"{prompt}\n\nFinancial Disclosure:\n{analysis_text}"

    response = client.chat.completions.create(

        model=FINE_TUNED_MODEL,

        messages=[

            {"role":"system","content":"Financial simplification assistant."},

            {"role":"user","content":full_prompt}

        ]

    )

    output = response.choices[0].message.content

    ft_results[f"Prompt_{i}"] = output


    print("\nFT PROMPT",i)

    print(output)


FT PROMPT 1
This document provides information about a company's financial performance and position. The company's fiscal year is 2025, and it has various financial obligations and assets listed.

FT PROMPT 2
The company had positive revenue growth. It faces risks related to long-term debt and lease obligations.

FT PROMPT 3
Financial Disclosure:
This document provides information about a company's finances. It shows details like revenue and debt.

FT PROMPT 4
The company faces risks related to competition, changing technology, and economic conditions. Trends include growing demand for its products and expansion into new markets.

FT PROMPT 5
This disclosure does not provide information about the company's performance. It includes technical details about the XBRL document format and metadata.


Also doing the measureables for readability using Flesch Reading Ease and Flesch-Kincaid grade level as I did before. This will help when I compare them later to see the differences and if the fine-tuning helped.

In [42]:
ft_readability = {}

print("\nFINE TUNE READABILITY")

for key,text in ft_results.items():

    reading = textstat.flesch_reading_ease(text)

    grade = textstat.flesch_kincaid_grade(text)

    ft_readability[key] = {

        "reading":reading,

        "grade":grade

    }

    print(key,reading,grade)


FINE TUNE READABILITY
Prompt_1 21.18615384615387 13.533846153846156
Prompt_2 45.377500000000026 8.917500000000004
Prompt_3 24.031029411764735 12.019117647058824
Prompt_4 21.930000000000035 13.181666666666665
Prompt_5 26.97750000000002 12.105


Want to make sure the fine-tuning doesn't have any of the keywords for investment advice done before. This is key since the fine-tuning in my mind would fail if it added any of this information. I added this later because I initially just assumed it would do this, but you shouldn't assume.

In [45]:

ft_advice = {}

print("\nFINE TUNE ADVICE CHECK")

for key,text in ft_results.items():

    violations = [

        word for word in advice_keywords

        if word in text.lower()

    ]

    ft_advice[key] = violations


    print(key,violations)


FINE TUNE ADVICE CHECK
Prompt_1 []
Prompt_2 []
Prompt_3 []
Prompt_4 []
Prompt_5 []


Finally, compared the readability, advice that does not have, and investment language, and showed each prompt to see the differences. Each time it was run, I could get some investment language or words in the baseline, but I never got any in the fine-tuned outputs. The readability level went down for each prompt. The fine-tuning is working to make it easier for the average person to understand what the information is saying.

In [48]:
print("\nFINAL COMPARISON")

for key in baseline_results:
    print("\n"+"="*60)
    print(key)
    print("="*60)

    print("\nREADABILITY")
    print("Baseline Grade:",baseline_readability[key]["grade"])
    print("FineTune Grade:",ft_readability[key]["grade"])


    print("\nADVICE")
    print("Baseline:",baseline_advice[key])
    print("FineTune:",ft_advice[key])


    print("\nBASELINE OUTPUT")
    print(baseline_results[key][:500])

    print("\nFINE TUNE OUTPUT")
    print(ft_results[key][:500])


FINAL COMPARISON

Prompt_1

READABILITY
Baseline Grade: 16.216666666666665
FineTune Grade: 13.533846153846156

ADVICE
Baseline: []
FineTune: []

BASELINE OUTPUT
This financial disclosure document contains information about a company's financial activities and performance. It includes details such as revenue expectations, debt obligations, asset values, and liabilities. The document is structured using a specific format called XBRL, which helps organize and present financial data for analysis and reporting purposes.

FINE TUNE OUTPUT
This document provides information about a company's financial performance and position. The company's fiscal year is 2025, and it has various financial obligations and assets listed.

Prompt_2

READABILITY
Baseline Grade: 21.676666666666666
FineTune Grade: 8.917500000000004

ADVICE
Baseline: []
FineTune: []

BASELINE OUTPUT
The financial performance and risks discussed in the document include revenue recognition, long-term debt, operating lease assets, fi


The fine-tuning worked. Readability decreased for each prompt; the outputs were much simpler and less jargon-heavy. There were no words in the fine-tuning that I could find anytime that would show this was for investment knowledge. The outputs make sense, which is key. I think there is room to get the readability level down even further with some tweaks. With the prompts, there is room to get into more detail on those.